In [ ]:
# Import libraries
import numpy as np
import tensorflow as tf
from sionna.rt import load_scene, ITURadioMaterial, Transmitter, Receiver, PlanarArray, RadioMapSolver, PathSolver
import matplotlib.pyplot as plt

In [ ]:
# Import scene from blender mitsuba export (.xml)
scene = load_scene("wall.xml")
print(f"Objects in scene: {list(scene.objects.keys())}")

In [ ]:
# Define array for both Tx and Rx
scene.tx_array = PlanarArray(num_rows=1, 
                             num_cols=1,
                             vertical_spacing=0.5,
                             horizontal_spacing=0.5,
                             pattern="tr38901",
                             polarization="V")

scene.rx_array = PlanarArray(num_rows=1, 
                             num_cols=1,
                             vertical_spacing=0.5,
                             horizontal_spacing=0.5,
                             pattern="iso",
                             polarization="V")

In [ ]:
# Define Frequency
carrier_freq = 3.5e9 # In Hz
scene.frequency = carrier_freq
print(f"Scene loaded. Frequency: {scene.frequency/1e9} GHz")

In [ ]:
# Add material to scene
scene.add(ITURadioMaterial(
    name="Concrete",
    itu_type="concrete",
    thickness=0.2
))

scene.add(ITURadioMaterial(
    name="Brick",
    itu_type="brick",
    thickness=0.2
))

scene.add(ITURadioMaterial(
    name="Wood",
    itu_type="wood",
    thickness=0.2
))

scene.add(ITURadioMaterial(
    name="Glass",
    itu_type="glass",
    thickness=0.2
))

scene.add(ITURadioMaterial(
    name="Plywood",
    itu_type="plywood",
    thickness=0.2
))

scene.add(ITURadioMaterial(
    name="Marble",
    itu_type="marble",
    thickness=0.2
))

scene.add(ITURadioMaterial(
    name="Metal",
    itu_type="metal",
    thickness=0.2
))

materials = [
    "Concrete",
    "Brick",
    "Wood",
    "Glass",
    "Plywood",
    "Marble",
    "Metal"
]

# Check if all material can be swapped for the wall
for m in materials:
    scene.objects["Cube"].radio_material = m
    print("Check:", m)

print(f"Radio Materials in scene: {list(scene.radio_materials.keys())}")

In [ ]:
# Deploy Tx and Rx location
tx = Transmitter("tx", [0, -40, 1.5], [0, 0,0], power_dbm=0)
rx = Receiver("rx", [0, 40, 1.5], [0, 0, 0])

# Point Tx and Rx towards each other
tx.look_at(rx.position)
rx.look_at(tx.position)

scene.add(tx)
scene.add(rx)

print (f'Tx: {tx}')
print (f'Rx: {rx}')

In [ ]:
# Preview scene before start processing

# Select material
scene.objects["Cube"].radio_material = "Concrete"

###---Choose between Radio Map or Paths

rm_solver = RadioMapSolver()

rm = rm_solver(
        scene,
        max_depth=10, # Number of maximum interation allowed per paths
        samples_per_tx=10**7, # Number of rays launch in transmitter
        cell_size=(1, 1),
        center=[0, 0, 0],
        size=[40, 100], # Size of area to simulate (meter)
        orientation=[0, 0, 0]
    )

scene.preview(
    radio_map=rm,

    rm_metric="rss",
    rm_db_scale=True,

    show_devices=True,
    show_orientations=True,

    point_picker=False
)

###---Choose between Radio Map or Paths

# p_solver = PathSolver()

# paths = p_solver(
#     scene,
#     max_depth=10,
#     samples_per_src=10**6
# )

# scene.preview(
#     paths=paths,

#     show_devices=True,
#     show_orientations=True,

#     point_picker=False
# )

In [ ]:
# Plot radio map

rm_solver = RadioMapSolver()

for m in materials:

    scene.objects["Cube"].radio_material = m
    print("Material:", m)

    rm = rm_solver(
        scene,
        max_depth=10,
        samples_per_tx=10**7,
        cell_size=(1, 1),
        center=[0, 0, 0],
        size=[40, 100],
        orientation=[0, 0, 0]
    )

    plt.figure(figsize=(8, 10))

    # Show radio map
    rm.show(metric="rss")

    # Better title
    plt.title(
        f"Material: {m}\n"
        f"Frequency: {carrier_freq/1e9}GHz",
        fontsize=9,
        pad=15
    )

    # Axis labels
    plt.xlabel("X Position (m)", fontsize=11)
    plt.ylabel("Y Position (m)", fontsize=11)

    # Fixed RSS range
    plt.clim(-160, -40)

    # Better grid
    plt.grid(
        alpha=0.2,
        linestyle="--"
    )

    plt.tight_layout()

    plt.show()

In [ ]:
# Path Analysis

p_solver = PathSolver()

valid_materials = []

rss_results = []
num_paths_results = []
strongest_results = []
delay_results = []

for m in materials:

    scene.objects["Cube"].radio_material = m
    print("Material:", m)

    paths = p_solver(
        scene,
        max_depth=10,
        samples_per_src=10**6
    )

    a = paths.a[0].numpy().flatten() # Path coeficients (amplitude + phase)
    tau = paths.tau[0].numpy().flatten() # Path delay

    power = np.abs(a)**2 # Power = V squared or I squared assumed if R is 0

    # skip empty results
    if power.size == 0:
        print(f"No paths for {m}")
        continue

    # RSS (total received power)
    rss_val = 10 * np.log10(np.sum(power)) + tx.power_dbm

    # strongest path
    strongest_val = 10 * np.log10(np.max(power)) + tx.power_dbm

    # number of paths
    num_paths = power.size

    # Plot Delay per Path

    plt.figure(figsize=(5, 5))

    plt.stem(
        tau * 1e9,
        10 * np.log10(power) + tx.power_dbm
    )

    plt.title(
        f"Power Delay Profile\n"
        f"Material: {m}\n"
        f"Frequency: {carrier_freq/1e9}GHz",
        fontsize=9,
        pad=15
    )
    plt.xlabel("Delay (ns)")
    plt.ylabel("RSS per Path (dBm)")
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

    valid_materials.append(m)
    rss_results.append(float(np.asarray(rss_val).item()))
    strongest_results.append(float(np.asarray(strongest_val).item()))
    num_paths_results.append(int(num_paths))

# Plot Summary

def bar_plot(data, title, ylabel):
    plt.figure(figsize=(5, 5))
    plt.bar(valid_materials, data)
    plt.title(title)
    plt.ylabel(ylabel)
    plt.xticks(rotation=45)
    plt.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.show()

bar_plot(rss_results, "Total RSS in Receiver", "dBm")
bar_plot(num_paths_results, "Number of Paths", "Count")
bar_plot(strongest_results, "Strongest Path RSS", "dBm")
